# Two-dimensional dijet unfolding of data

Unfold the combined-orientation trigger-selected data distributions used by `06_data_dijet_distributions.ipynb` in flattened $p_T^{ave}$–$\eta_{CM}$ space. The combined embedding sample supplies the response matrix, reconstructed-bin purity (fake correction), and truth-bin efficiency (miss correction). The default is 12 Bayesian iterations.

Each data interval is taken from exactly one dataset: MinimumBias for 50–80 GeV, Jet60 for 80–100 GeV, Jet80 for 100–120 GeV, and Jet100 above 120 GeV. No trigger spectra are merged within an interval.

<!-- detailed-workflow-guide -->

### Detailed workflow and interpretation

The combined-orientation embedding response is flattened with global-bin ordering `[pTave block][eta bin]`, retaining migrations between all selected pTave intervals. The inclusive combined-data vector $m_i$ is corrected with the fixed embedding purity $P_i=m_i^{matched}/m_i^{inclusive}$, unfolded with the matched-event migration matrix, and divided by the embedding efficiency $\epsilon_j=t_j^{matched}/t_j^{inclusive}$:

$$m_i^{data} \rightarrow P_i m_i^{data} \rightarrow U_{12}[P m] \rightarrow \hat t_j^{matched}/\epsilon_j.$$

Purity and efficiency uncertainties are not added independently here; they are fixed response inputs and belong in coherent embedding statistical/systematic variations. The saved covariance is the RooUnfold covariance transformed by the efficiency correction. A forward-folded comparison checks consistency at reconstructed level. The unfolded data have no truth closure target.

## 1. Environment and imports

In [1]:
# Cell role: initialize the reproducible Python/ROOT environment and shared helpers.
# Interpretation: No physics histogram is modified here; ROOT ownership is configured before files open.
# The preceding Markdown gives the equations and physics assumptions for this step.
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys

PROJECT_ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
                     if (candidate / 'CMakeLists.txt').is_file()
                     and (candidate / 'hist_analysis').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter from the jetAnalysis repository root')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root, load_roounfold
ROOT = load_root(batch=True)
ROOUNFOLD_ROOT, ROOUNFOLD_LIBRARY = load_roounfold(ROOT, project_root=PROJECT_ROOT)
try:
    import RooUnfold
except ImportError as exc:
    raise ImportError(f'Unable to import RooUnfold after loading {ROOUNFOLD_LIBRARY}') from exc
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import STANDARD_DIJET_ETA_CUT_INDEX
from hist_analysis.python.histogram_io import (
    load_histogram, resolve_combined_file, resolve_data_file, resolve_direction_file,
)
from hist_analysis.python.plotting import draw_closure
from hist_analysis.python.unfolding import (
    UnfoldingInputKeys, apply_efficiency_correction, as_pt_intervals,
    build_roounfold_response, calculate_closure_metrics,
    calculate_response_diagnostics, flatten_pt_eta_projections,
    flatten_sparse_response, forward_fold_truth, load_unfolding_inputs,
    prepare_factorized_corrections, project_eta_by_pt, unfold_bayes,
    unflatten_to_eta_projections, validate_response_accounting,
    write_unfolding_output,
)
from hist_analysis.python.unfolding_plots import (
    draw_flattened_response, draw_response_components, draw_response_matrix,
)

## 2. Configuration

In [2]:
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
GENERATOR = 'embedding'
DATA_DIRECTIONS = ('combined',)  # response, fakes, misses, and data
FILE_STEM = 'jetId'
DATA_SELECTION = 'jetId'  # jetId, trkMax, or noSel
N_ITERATIONS = 12
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.3, 2.4, 3.0)
ETA_CUT_INDEX = STANDARD_DIJET_ETA_CUT_INDEX
PTAVE_BINS_BY_DATASET = {
    'MinimumBias': ((50, 60), (60, 70), (70, 80)),
    'Jet60': ((80, 90), (90, 100)),
    'Jet80': ((100, 110), (110, 120)),
    'Jet100': ((120, 130), (130, 140), (140, 150), (150, 160),
               (160, 180), (180, 200), (200, 250), (250, 300), (300, 500)),
}
RESPONSE_SCALE = 1.0e12
RATIO_Y_RANGE = (0.75, 1.25)
DRAW_GRID = True
SAVE_PNG = False
DATA_DIR = Path(os.environ.get('PPB_DATA_DIR', BASE_DIR / 'exp'))
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_UNFOLD2D_DATA_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis/output/unfold2D_data',
)) / DATA_SELECTION

if GENERATOR != 'embedding':
    raise ValueError('This data workflow requires embedding response inputs')
if DATA_SELECTION not in {'jetId', 'trkMax', 'noSel'}:
    raise ValueError(f'Unsupported DATA_SELECTION={DATA_SELECTION!r}')
if not DATA_DIRECTIONS or any(value not in {'Pbgoing', 'pgoing', 'combined'} for value in DATA_DIRECTIONS):
    raise ValueError(f'Unsupported DATA_DIRECTIONS={DATA_DIRECTIONS!r}')
if len(set(DATA_DIRECTIONS)) != len(DATA_DIRECTIONS):
    raise ValueError('DATA_DIRECTIONS contains duplicates')
if N_ITERATIONS < 1 or RESPONSE_SCALE <= 0.0:
    raise ValueError('N_ITERATIONS and RESPONSE_SCALE must be positive')
if not 0 <= ETA_CUT_INDEX < len(ETA_CUTS):
    raise IndexError(f'Invalid ETA_CUT_INDEX={ETA_CUT_INDEX}')

PTAVE_INTERVALS = as_pt_intervals(tuple(
    interval for intervals in PTAVE_BINS_BY_DATASET.values() for interval in intervals
))
TRIGGER_BY_INTERVAL = {interval: dataset
                       for dataset, intervals in PTAVE_BINS_BY_DATASET.items()
                       for interval in intervals}
if len(PTAVE_INTERVALS) != 16 or len(TRIGGER_BY_INTERVAL) != 16:
    raise ValueError('Expected the 16 unique pTave intervals from notebook 06')
ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]
ETA_UNFOLDING_RANGE = (-ETA_CUT, ETA_CUT)
eta_cut_tag = f'{ETA_CUT:g}'.replace('.', 'p')
print(f'{len(PTAVE_INTERVALS)} intervals; {N_ITERATIONS} Bayesian iterations')

16 intervals; 12 Bayesian iterations


## 3. Load and flatten the embedding response

The combined embedding production supplies the response, misses, and fakes. The response marginals, rather than the explicit miss/fake histograms alone, define the effective corrections after restricting the unfolding domain; this includes migrations across the selected pTave and eta boundaries.

In [3]:
# Cell role: resolve embedding inputs and construct one validated response model per direction.
# Interpretation: Matrix marginals define matched spectra; residuals define effective misses and fakes.
# The preceding Markdown gives the equations and physics assumptions for this step.
response_models = {}
for direction in DATA_DIRECTIONS:
    embedding_file = (resolve_combined_file(BASE_DIR, GENERATOR, FILE_STEM)
                      if direction == 'combined' else
                      resolve_direction_file(BASE_DIR, GENERATOR, direction, FILE_STEM))
    keys = UnfoldingInputKeys(
        truth=f'hGenDijetPtEtaCM_{ETA_CUT_INDEX}',
        measured=f'hRecoDijetPtEtaCMJerDefExtraUnfold_{ETA_CUT_INDEX}',
        response=f'hGenDijetPtEtaCMVsRecoJerDefExtraPtEtaCM_{ETA_CUT_INDEX}',
        miss=f'hGenDijetPtEtaCMMissJerDefExtra_{ETA_CUT_INDEX}',
        fake=f'hRecoDijetPtEtaCMFakeJerDefExtra_{ETA_CUT_INDEX}',
    )
    inputs = load_unfolding_inputs(embedding_file, keys)
    projection_options = {'eta_range': ETA_UNFOLDING_RANGE}
    truth_by_pt = project_eta_by_pt(inputs.truth, PTAVE_INTERVALS,
                                     name_prefix=f'h{direction}TrainingTruth', **projection_options)
    reco_by_pt = project_eta_by_pt(inputs.measured, PTAVE_INTERVALS,
                                    name_prefix=f'h{direction}TrainingReco', **projection_options)
    miss_by_pt = project_eta_by_pt(inputs.miss, PTAVE_INTERVALS,
                                    name_prefix=f'h{direction}TrainingMiss', **projection_options)
    fake_by_pt = project_eta_by_pt(inputs.fake, PTAVE_INTERVALS,
                                    name_prefix=f'h{direction}TrainingFake', **projection_options)
    training_truth, layout = flatten_pt_eta_projections(
        truth_by_pt, name=f'h{direction}TrainingTruth', pt_bins=PTAVE_INTERVALS)
    training_reco, _ = flatten_pt_eta_projections(
        reco_by_pt, name=f'h{direction}TrainingReco', layout=layout)
    explicit_miss, _ = flatten_pt_eta_projections(
        miss_by_pt, name=f'h{direction}ExplicitMiss', layout=layout)
    explicit_fake, _ = flatten_pt_eta_projections(
        fake_by_pt, name=f'h{direction}ExplicitFake', layout=layout)
    response_matrix, _ = flatten_sparse_response(
        inputs.response, PTAVE_INTERVALS, name=f'h{direction}ResponseEtaCM',
        layout=layout, eta_range=ETA_UNFOLDING_RANGE)
    diagnostics = calculate_response_diagnostics(
        response_matrix, training_truth, training_reco,
        explicit_miss=explicit_miss, explicit_fake=explicit_fake)
    accounting = validate_response_accounting(
        response_matrix, training_truth, training_reco, diagnostics)
    matched_response = build_roounfold_response(
        RooUnfold, diagnostics.matched_truth, diagnostics.matched_measured,
        response_matrix, scale=RESPONSE_SCALE, require_fakes=False,
        name=f'{direction}MatchedResponse')
    inclusive_response = build_roounfold_response(
        RooUnfold, training_truth, training_reco, response_matrix,
        diagnostics=diagnostics, scale=RESPONSE_SCALE, require_fakes=True,
        name=f'{direction}InclusiveResponse')
    response_models[direction] = {
        'file': embedding_file, 'keys': keys, 'layout': layout,
        'truth_by_pt': truth_by_pt, 'reco_by_pt': reco_by_pt,
        'training_truth': training_truth, 'training_reco': training_reco,
        'explicit_miss': explicit_miss, 'explicit_fake': explicit_fake,
        'response_matrix': response_matrix, 'diagnostics': diagnostics,
        'accounting': accounting, 'matched_response': matched_response,
        'inclusive_response': inclusive_response,
    }
    print(f'{direction}: {embedding_file}')
    print(f'  efficiency={accounting.global_efficiency:.4f}, '
          f'fake fraction={accounting.global_fake_fraction:.4f}, '
          f'sparsity={accounting.response_sparsity:.4f}')

combined: /Users/gnigmat/cernbox/ana/pPb8160/embedding/embedding_jetId.root
  efficiency=0.8100, fake fraction=0.2357, sparsity=0.9756


## 4. Build trigger-stitched data vectors

The data histogram for each pTave block comes from the same trigger dataset and half-open interval used in notebook 06. Contents and data statistical errors are preserved when the blocks are flattened.

In [4]:
# Cell role: load data and flatten the configured trigger-specific pTave projections.
# Interpretation: Each pTave interval is supplied by one dataset; no cross-trigger normalization is applied.
# The preceding Markdown gives the equations and physics assumptions for this step.
data_inputs = {}
for direction in DATA_DIRECTIONS:
    model = response_models[direction]
    trigger_maps = {}
    data_by_pt = []
    data_files = {}
    for pt_index, interval in enumerate(PTAVE_INTERVALS):
        trigger = TRIGGER_BY_INTERVAL[interval]
        if trigger not in trigger_maps:
            data_file = resolve_data_file(DATA_DIR, trigger, direction, DATA_SELECTION)
            data_files[trigger] = data_file
            trigger_maps[trigger] = load_histogram(str(data_file), 'hRecoDijetPtEtaCM')
        projection = project_eta_by_pt(
            trigger_maps[trigger], (interval,),
            name_prefix=f'h{direction}Data{trigger}Pt{pt_index}',
            eta_range=ETA_UNFOLDING_RANGE)[0]
        data_by_pt.append(projection)
    data_reco, _ = flatten_pt_eta_projections(
        data_by_pt, name=f'h{direction}TriggerStitchedData', layout=model['layout'])
    data_inputs[direction] = {
        'files': data_files, 'by_pt': data_by_pt, 'reco': data_reco,
    }
    print(f'{direction} data files:')
    for trigger, path in data_files.items():
        print(f'  {trigger}: {path}')

combined data files:
  MinimumBias: /Users/gnigmat/cernbox/ana/pPb8160/exp/mb_ak4_jetId.root
  Jet60: /Users/gnigmat/cernbox/ana/pPb8160/exp/jet60_ak4_jetId.root
  Jet80: /Users/gnigmat/cernbox/ana/pPb8160/exp/jet80_ak4_jetId.root
  Jet100: /Users/gnigmat/cernbox/ana/pPb8160/exp/jet100_ak4_jetId.root


## 5. Purity correction, Bayesian unfolding, and efficiency correction

The embedding corrections are treated as fixed. Therefore the measured covariance propagated here contains the data statistical uncertainty; response statistical and detector-model uncertainties require dedicated variations.

In [5]:
# Cell role: unfold every direction with the direction-matched embedding response.
# Interpretation: Data are corrected in the order purity, matched migration, then efficiency.
# The preceding Markdown gives the equations and physics assumptions for this step.
unfolded_results = {}
for direction in DATA_DIRECTIONS:
    model = response_models[direction]
    data = data_inputs[direction]
    diagnostics = model['diagnostics']
    factorized = prepare_factorized_corrections(
        data['reco'], model['training_reco'], diagnostics.matched_measured,
        model['training_truth'], diagnostics.matched_truth,
        name_prefix=f'h{direction}DataFactorized')
    matched = unfold_bayes(
        RooUnfold, model['matched_response'], factorized.measured_signal,
        iterations=N_ITERATIONS, handle_fakes=False,
        name=f'h{direction}DataUnfoldedMatched')
    unfolded, covariance = apply_efficiency_correction(
        matched.histogram, matched.covariance, factorized.efficiency,
        name=f'h{direction}DataUnfolded')
    unfolded_by_pt = unflatten_to_eta_projections(
        unfolded, model['truth_by_pt'], model['layout'],
        name_prefix=f'h{direction}DataUnfoldedEtaCM')
    refold = forward_fold_truth(
        model['inclusive_response'], unfolded, diagnostics=diagnostics,
        add_fakes=True, fake_normalization='matched_fraction',
        name=f'h{direction}DataForwardFolded')
    refold_metrics = calculate_closure_metrics(refold.histogram, data['reco'])
    unfolded_results[direction] = {
        'factorized': factorized, 'matched': matched, 'unfolded': unfolded,
        'covariance': covariance, 'by_pt': unfolded_by_pt,
        'refold': refold, 'refold_metrics': refold_metrics,
    }
    print(f'{direction}: unfolded {unfolded.Integral():.6g} entries')
    print(f'  refold/data integral ratio={refold_metrics.integral_ratio:.6f}, '
          f'yield-weighted L1={refold_metrics.relative_l1:.2%}')

combined: unfolded 1.00882e+07 entries
  refold/data integral ratio=1.000613, yield-weighted L1=17.69%


## 6. Response and reconstructed-level validation plots

The response diagnostics show the embedding model. The forward-folded unfolded spectrum is compared with the original data in reconstructed global bins; this is a consistency check, not an independent closure test.

In [6]:
# Cell role: draw response diagnostics and forward-folded data consistency checks.
# Interpretation: Refold/data agreement tests the configured inverse model at reconstructed level.
# The preceding Markdown gives the equations and physics assumptions for this step.
diagnostic_canvases = {}
for direction in DATA_DIRECTIONS:
    model = response_models[direction]
    result = unfolded_results[direction]
    data_reco = data_inputs[direction]['reco']
    tag = f'embedding_{direction}_data_eta_{eta_cut_tag}_iter_{N_ITERATIONS}'
    direction_dir = OUTPUT_DIR / direction
    flattened_canvas = draw_flattened_response(
        {'gen': ('Embedding Gen', model['training_truth']),
         'reco': ('Embedding Reco', model['training_reco']),
         'miss': ('Explicit miss', model['explicit_miss']),
         'fake': ('Explicit fake', model['explicit_fake'])},
        model['response_matrix'], annotations=('Embedding', direction, f'|#eta_{{CM}}| < {ETA_CUT:g}'),
        plot_miss_and_fakes=True, output=direction_dir / f'{tag}_flattened_response.pdf',
        save_png=SAVE_PNG, grid=DRAW_GRID, canvas_name=f'canvas_{direction}_flattened')
    matrix_canvas = draw_response_matrix(
        model['response_matrix'], annotations=('Embedding', direction, f'{len(PTAVE_INTERVALS)} p_{{T}}^{{ave}} intervals'),
        output=direction_dir / f'{tag}_response_matrix.pdf', save_png=SAVE_PNG,
        grid=False, log_z=True, canvas_name=f'canvas_{direction}_response_matrix')
    components_canvas = draw_response_components(
        explicit_miss=model['explicit_miss'], boundary_miss=model['diagnostics'].boundary_miss,
        effective_miss=model['diagnostics'].effective_miss, explicit_fake=model['explicit_fake'],
        boundary_fake=model['diagnostics'].boundary_fake, effective_fake=model['diagnostics'].effective_fake,
        annotations=('Embedding', direction), output=direction_dir / f'{tag}_response_components.pdf',
        save_png=SAVE_PNG, grid=DRAW_GRID, canvas_name=f'canvas_{direction}_components')
    refold_canvas, refold_ratios = draw_closure(
        {'Data': data_reco, 'Forward-folded unfolded data': result['refold'].histogram},
        nominal='Data', title='', x_title='global measured bin', y_title='Entries',
        ratio_range=RATIO_Y_RANGE, draw_nominal_ratio=False,
        annotations=(direction, f'Bayesian iterations: {N_ITERATIONS}'),
        output=direction_dir / f'{tag}_refolded_to_data.pdf', save_png=SAVE_PNG,
        grid=DRAW_GRID, canvas_name=f'canvas_{direction}_refold')
    refold_ratio = refold_ratios['Forward-folded unfolded data']
    refold_ratio.SetName(f'h{direction}ForwardFoldedToData')
    diagnostic_canvases[direction] = (flattened_canvas, matrix_canvas, components_canvas, refold_canvas, refold_ratio)
    display(matrix_canvas)
    display(components_canvas)
    display(refold_canvas)

Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D_data/jetId/combined/embedding_combined_data_eta_1p9_iter_12_flattened_response.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D_data/jetId/combined/embedding_combined_data_eta_1p9_iter_12_response_matrix.pdf has been created


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D_data/jetId/combined/embedding_combined_data_eta_1p9_iter_12_response_components.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D_data/jetId/combined/embedding_combined_data_eta_1p9_iter_12_refolded_to_data.pdf has been created


## 7. Data before and after unfolding in every pTave interval

For every trigger-specific interval, compare reconstructed data before unfolding with data after purity, migration, and efficiency corrections. One canvas preserves the unnormalized yields; a second compares independently normalized $1/N\,dN/d\eta_{CM}$ shapes. Ratios use standard independent-error propagation for display only; quantitative use of the unfolded result must retain its saved covariance.

In [7]:
# Cell role: compare data before and after unfolding in yield and unit-area shape for every interval.
# Interpretation: Shape normalization is performed on clones; unfolded-bin correlations live in the saved covariance.
# The preceding Markdown gives the equations and physics assumptions for this step.
before_after_canvases = {}
before_after_ratios = {}
normalized_data_before_by_direction = {}
normalized_unfolded_by_direction = {}
for direction in DATA_DIRECTIONS:
    result = unfolded_results[direction]
    direction_dir = OUTPUT_DIR / direction
    canvases = []
    ratios = []
    normalized_before = []
    normalized_after = []
    for pt_index, ((low, high), before, after) in enumerate(
        zip(PTAVE_INTERVALS, data_inputs[direction]['by_pt'], result['by_pt'])):
        trigger = TRIGGER_BY_INTERVAL[(low, high)]
        annotations = (f'{direction}, {trigger}',
                       f'{low:g} #leq p_{{T}}^{{ave}} < {high:g} GeV',
                       f'Bayesian iterations: {N_ITERATIONS}')
        yield_canvas, yield_ratios = draw_closure(
            {'Before unfolding': before, 'After unfolding': after},
            nominal='Before unfolding', title='', x_title='#eta_{CM}^{dijet}',
            y_title='Entries', ratio_range=RATIO_Y_RANGE, draw_nominal_ratio=False,
            x_range=ETA_UNFOLDING_RANGE, annotations=annotations,
            output=direction_dir / f'data_before_after_yield_pt_{low:g}_{high:g}.pdf',
            save_png=SAVE_PNG, grid=DRAW_GRID,
            canvas_name=f'canvas_{direction}_before_after_yield_pt{pt_index}')
        yield_ratio = yield_ratios['After unfolding']
        yield_ratio.SetName(f'h{direction}DataAfterToBefore_pt{pt_index}')
        before_density = before.Clone(f'h{direction}DataBeforeDensity_pt{pt_index}')
        after_density = after.Clone(f'h{direction}DataAfterDensity_pt{pt_index}')
        for label, density in (('before', before_density), ('after', after_density)):
            density.SetDirectory(0)
            area = density.Integral('width')
            if area <= 0.0:
                raise ValueError(f'Cannot normalize {label} histogram for interval {(low, high)}: area={area}')
            density.Scale(1.0 / area)
        shape_canvas, shape_ratios = draw_closure(
            {'Before unfolding': before_density, 'After unfolding': after_density},
            nominal='Before unfolding', title='', x_title='#eta_{CM}^{dijet}',
            y_title='1/N dN/d#eta_{CM}^{dijet}', ratio_range=RATIO_Y_RANGE,
            draw_nominal_ratio=False, x_range=ETA_UNFOLDING_RANGE, annotations=annotations,
            output=direction_dir / f'data_before_after_shape_pt_{low:g}_{high:g}.pdf',
            save_png=SAVE_PNG, grid=DRAW_GRID,
            canvas_name=f'canvas_{direction}_before_after_shape_pt{pt_index}')
        shape_ratio = shape_ratios['After unfolding']
        shape_ratio.SetName(f'h{direction}DataAfterToBeforeShape_pt{pt_index}')
        canvases.extend((yield_canvas, shape_canvas))
        ratios.extend((yield_ratio, shape_ratio))
        normalized_before.append(before_density); normalized_after.append(after_density)
    before_after_canvases[direction] = canvases
    before_after_ratios[direction] = ratios
    normalized_data_before_by_direction[direction] = normalized_before
    normalized_unfolded_by_direction[direction] = normalized_after
before_after_canvases

Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D_data/jetId/combined/data_before_after_yield_pt_50_60.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D_data/jetId/combined/data_before_after_shape_pt_50_60.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D_data/jetId/combined/data_before_after_yield_pt_60_70.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D_data/jetId/combined/data_before_after_shape_pt_60_70.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D_data/jetId/combined/data_before_after_yield_pt_70_80.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D_data/jetId/combined/data

{'combined': [<cppyy.gbl.TCanvas object at 0xb3a308c00>,
  <cppyy.gbl.TCanvas object at 0xb3a6d0400>]}

Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D_data/jetId/combined/data_before_after_yield_pt_180_200.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D_data/jetId/combined/data_before_after_shape_pt_180_200.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D_data/jetId/combined/data_before_after_yield_pt_200_250.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D_data/jetId/combined/data_before_after_shape_pt_200_250.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D_data/jetId/combined/data_before_after_yield_pt_250_300.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D_data/jetId/com

## 8. Save unfolding artifacts

One ROOT file per direction stores the original stitched data, fixed embedding corrections, unfolded result, transformed covariance, refolded validation objects, per-interval yields and presentation densities, plus complete input metadata.

In [8]:
# Cell role: persist data unfolding products, covariance, response, and provenance.
# Interpretation: ROOT outputs preserve unnormalized yields; density clones supplement them.
# The preceding Markdown gives the equations and physics assumptions for this step.
output_files = {}
for direction in DATA_DIRECTIONS:
    model = response_models[direction]
    data = data_inputs[direction]
    result = unfolded_results[direction]
    factorized = result['factorized']
    refold_ratio = diagnostic_canvases[direction][-1]
    output_file = OUTPUT_DIR / direction / (
        f'embedding_{direction}_unfold2D_data_eta_{eta_cut_tag}_iter_{N_ITERATIONS}.root')
    write_unfolding_output(
        output_file,
        histograms=(
            data['reco'], model['training_truth'], model['training_reco'],
            model['explicit_miss'], model['explicit_fake'], model['response_matrix'],
            model['diagnostics'].matched_truth, model['diagnostics'].matched_measured,
            model['diagnostics'].effective_miss, model['diagnostics'].effective_fake,
            model['diagnostics'].boundary_miss, model['diagnostics'].boundary_fake,
            factorized.measured_signal, factorized.purity, factorized.efficiency,
            result['matched'].histogram, result['unfolded'],
            result['refold'].folded_signal, result['refold'].fake,
            result['refold'].histogram, refold_ratio,
            *data['by_pt'], *result['by_pt'],
            *normalized_data_before_by_direction[direction],
            *normalized_unfolded_by_direction[direction],
            *before_after_ratios[direction],
        ),
        covariance=result['covariance'], covariance_name='hDataUnfoldedCovariance',
        response=model['matched_response'].response,
        metadata={
            'embedding_file': str(model['file']),
            'response_keys': model['keys'].__dict__,
            'data_files': {key: str(value) for key, value in data['files'].items()},
            'generator': GENERATOR, 'direction': direction,
            'data_selection': DATA_SELECTION, 'iterations': N_ITERATIONS,
            'eta_cut': ETA_CUT, 'eta_unfolding_range': ETA_UNFOLDING_RANGE,
            'pt_ave_bins': PTAVE_INTERVALS,
            'trigger_by_interval': {f'{low:g}_{high:g}': TRIGGER_BY_INTERVAL[(low, high)]
                                    for low, high in PTAVE_INTERVALS},
            'response_scale': RESPONSE_SCALE,
            'unfolding_method': 'factorized_purity_migration_efficiency',
            'purity_source': 'combined embedding',
            'efficiency_source': 'combined embedding',
            'response_sparsity': model['accounting'].response_sparsity,
            'global_efficiency': model['accounting'].global_efficiency,
            'global_fake_fraction': model['accounting'].global_fake_fraction,
            'refold_integral_ratio': result['refold_metrics'].integral_ratio,
            'refold_relative_l1': result['refold_metrics'].relative_l1,
        },
    )
    output_files[direction] = output_file
    print(f'Wrote {output_file}')
output_files

Wrote /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D_data/jetId/combined/embedding_combined_unfold2D_data_eta_1p9_iter_12.root


{'combined': PosixPath('/Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/unfold2D_data/jetId/combined/embedding_combined_unfold2D_data_eta_1p9_iter_12.root')}